# Tema 22 — Encoder de imagen (CNN) + Encoder de texto (embeddings): un mini-CLIP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-11/Tema-22/Tema_22.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

Construimos un modelo **contrastivo** estilo **CLIP** en miniatura con TensorFlow/Keras:

- un **encoder de imagen** basado en **redes convolucionales (CNN)**, y
- un **encoder de texto** basado en **capas de *embedding***.

Ambos proyectan a un **espacio común** donde una imagen y su descripción quedan cerca. Entrenamos con una **pérdida contrastiva** (alinear imagen↔texto) sobre un dataset **sintético** de figuras de colores y, al final, hacemos **búsqueda texto→imagen** y medimos **Recall@K**.

## 1. Instalación de dependencias

Este notebook usa **TensorFlow**, **NumPy** y **Matplotlib**, que ya vienen preinstalados en **Google Colab**. En local, instala las dependencias con el `README.md` de la carpeta.

In [ ]:
# === Dependencias (en Colab ya vienen preinstaladas) ===
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print("Colab detectado: TensorFlow, NumPy y Matplotlib ya están disponibles.")
    # Si hiciera falta: !pip install -q tensorflow matplotlib numpy
else:
    print("Entorno local detectado. Asegúrate de tener: tensorflow, numpy, matplotlib (ver README.md).")

## 2. Importaciones y configuración

Fijamos una **semilla** (`SEED`) para reproducibilidad y definimos los hiperparámetros: tamaño de imagen, número de muestras por clase, tamaño de lote, dimensión del *embedding* y número de épocas.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# ============================================================
# Configuración
# ============================================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 64
NUM_SAMPLES_PER_CLASS = 40
BATCH_SIZE = 32
EMBED_DIM = 64
EPOCHS = 8
AUTOTUNE = tf.data.AUTOTUNE

## 3. Dataset sintético con semántica real

Generamos imágenes de **figuras** (cuadrado, círculo, triángulo) en tres **colores** (rojo, verde, azul). Cada imagen se asocia a una descripción tipo `"green triangle"`. La función `draw_shape()` dibuja la figura sobre un fondo oscuro y añadimos un poco de **ruido** para dar variación. Así obtenemos 9 clases (3 colores × 3 figuras) con una relación imagen↔texto real.

In [ ]:
colors = {
    "red":   np.array([1.0, 0.0, 0.0], dtype=np.float32),
    "green": np.array([0.0, 1.0, 0.0], dtype=np.float32),
    "blue":  np.array([0.0, 0.0, 1.0], dtype=np.float32),
}

shapes = ["square", "circle", "triangle"]

def draw_shape(color_name, shape_name, size=IMG_SIZE):
    image = np.ones((size, size, 3), dtype=np.float32) * 0.1
    color = colors[color_name]

    yy, xx = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2

    if shape_name == "square":
        mask = (xx > size * 0.25) & (xx < size * 0.75) & (yy > size * 0.25) & (yy < size * 0.75)

    elif shape_name == "circle":
        r = size * 0.22
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2

    elif shape_name == "triangle":
        # Triángulo simple
        x1, y1 = size * 0.5, size * 0.2
        x2, y2 = size * 0.2, size * 0.75
        x3, y3 = size * 0.8, size * 0.75

        denom = ((y2 - y3)*(x1 - x3) + (x3 - x2)*(y1 - y3))
        a = ((y2 - y3)*(xx - x3) + (x3 - x2)*(yy - y3)) / denom
        b = ((y3 - y1)*(xx - x3) + (x1 - x3)*(yy - y3)) / denom
        c = 1 - a - b
        mask = (a >= 0) & (b >= 0) & (c >= 0)

    image[mask] = color
    return image

images = []
texts = []

for color_name in colors.keys():
    for shape_name in shapes:
        caption = f"{color_name} {shape_name}"
        for _ in range(NUM_SAMPLES_PER_CLASS):
            img = draw_shape(color_name, shape_name)
            # Pequeño ruido para variación
            img = np.clip(img + np.random.normal(0, 0.02, img.shape), 0, 1).astype(np.float32)
            images.append(img)
            texts.append(caption)

images = np.array(images, dtype=np.float32)
texts = np.array(texts)

print("Imágenes:", images.shape)
print("Textos:", texts.shape)

## 4. Vectorización del texto

Convertimos cada descripción (dos palabras: color + figura) en una secuencia de **enteros** con `TextVectorization` (`output_sequence_length=2`). Estos IDs alimentarán el encoder de texto.

In [ ]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=50,
    output_mode="int",
    output_sequence_length=2
)
vectorizer.adapt(texts)

text_ids = vectorizer(texts)

## 5. Partición train/val y `tf.data`

Barajamos y separamos **80 % entrenamiento / 20 % validación**, y creamos *pipelines* eficientes de `tf.data` (con `shuffle`, `batch` y `prefetch`). Cada elemento es un diccionario `{"image": ..., "text": ...}`.

In [ ]:
num_samples = len(images)
indices = np.arange(num_samples)
np.random.shuffle(indices)

split = int(num_samples * 0.8)
train_idx = indices[:split]
val_idx = indices[split:]

train_images, val_images = images[train_idx], images[val_idx]
train_texts, val_texts = text_ids.numpy()[train_idx], text_ids.numpy()[val_idx]
val_texts_raw = texts[val_idx]

train_ds = tf.data.Dataset.from_tensor_slices(
    {"image": train_images, "text": train_texts}
).shuffle(500, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    {"image": val_images, "text": val_texts}
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

## 6. Encoder de imagen (CNN)

Una CNN compacta (bloques `Conv2D` + `MaxPooling2D`) que resume la imagen en un vector y lo proyecta al espacio de *embeddings* de dimensión `EMBED_DIM` mediante una capa `Dense`.

In [ ]:
def build_image_encoder():
    image_input = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.layers.Conv2D(16, 3, activation="relu")(image_input)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(32, 3, activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(64, 3, activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(EMBED_DIM)(x)
    return tf.keras.Model(image_input, x, name="image_encoder")

## 7. Encoder de texto (embeddings)

Convierte los IDs de las palabras en vectores con una capa `Embedding`, los promedia (`GlobalAveragePooling1D`) y los proyecta al **mismo** espacio de dimensión `EMBED_DIM` que el encoder de imagen.

In [ ]:
def build_text_encoder(vocab_size):
    text_input = tf.keras.Input(shape=(2,), dtype=tf.int64)
    x = tf.keras.layers.Embedding(vocab_size, 32)(text_input)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(EMBED_DIM)(x)
    return tf.keras.Model(text_input, x, name="text_encoder")

## 8. Modelo contrastivo y pérdida

`ContrastiveModel` combina ambos encoders. En cada lote:

1. normaliza los *embeddings* (L2),
2. calcula la matriz de **similitudes** imagen↔texto escalada por una `temperature` aprendible,
3. aplica una **pérdida contrastiva** simétrica (imagen→texto y texto→imagen): la diagonal (pares correctos) debe tener la mayor similitud.

Se sobreescriben `train_step` y `test_step` para controlar el entrenamiento a bajo nivel.

In [ ]:
class ContrastiveModel(tf.keras.Model):
    def __init__(self, image_encoder, text_encoder, temperature=0.07):
        super().__init__()
        self.image_encoder = image_encoder
        self.text_encoder = text_encoder
        self.temperature = tf.Variable(temperature, trainable=True, dtype=tf.float32)
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def compute_loss(self, img_emb, txt_emb):
        img_emb = tf.math.l2_normalize(img_emb, axis=1)
        txt_emb = tf.math.l2_normalize(txt_emb, axis=1)

        logits = tf.matmul(img_emb, txt_emb, transpose_b=True) / self.temperature
        labels = tf.range(tf.shape(logits)[0])

        loss_i2t = tf.keras.losses.sparse_categorical_crossentropy(
            labels, logits, from_logits=True
        )
        loss_t2i = tf.keras.losses.sparse_categorical_crossentropy(
            labels, tf.transpose(logits), from_logits=True
        )

        return (tf.reduce_mean(loss_i2t) + tf.reduce_mean(loss_t2i)) / 2.0

    def train_step(self, batch):
        images = batch["image"]
        texts = batch["text"]

        with tf.GradientTape() as tape:
            img_emb = self.image_encoder(images, training=True)
            txt_emb = self.text_encoder(texts, training=True)
            loss = self.compute_loss(img_emb, txt_emb)

        variables = (
            self.image_encoder.trainable_variables +
            self.text_encoder.trainable_variables +
            [self.temperature]
        )
        grads = tape.gradient(loss, variables)
        self.optimizer.apply_gradients(zip(grads, variables))

        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, batch):
        images = batch["image"]
        texts = batch["text"]

        img_emb = self.image_encoder(images, training=False)
        txt_emb = self.text_encoder(texts, training=False)
        loss = self.compute_loss(img_emb, txt_emb)

        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

## 9. Entrenamiento

Construimos los encoders, instanciamos el modelo contrastivo, lo compilamos con `Adam` y entrenamos durante `EPOCHS` épocas monitoreando la pérdida en validación.

In [ ]:
vocab_size = len(vectorizer.get_vocabulary())

image_encoder = build_image_encoder()
text_encoder = build_text_encoder(vocab_size)

model = ContrastiveModel(image_encoder, text_encoder)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3))

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1
)

## 10. Obtener *embeddings* de validación

Codificamos las imágenes y textos de validación y **normalizamos** (L2) para poder compararlos por producto punto (equivale a la similitud coseno).

In [ ]:
def encode_dataset(images_array, texts_array):
    img_emb = image_encoder(images_array, training=False)
    txt_emb = text_encoder(texts_array, training=False)

    img_emb = tf.math.l2_normalize(img_emb, axis=1).numpy()
    txt_emb = tf.math.l2_normalize(txt_emb, axis=1).numpy()
    return img_emb, txt_emb

val_img_emb, val_txt_emb = encode_dataset(val_images, val_texts)

## 11. Evaluación con Recall@K

Para cada texto de validación buscamos las imágenes más similares y medimos si la imagen correcta aparece entre las **K** primeras. `Recall@1` y `Recall@5` resumen la calidad del alineamiento aprendido.

In [ ]:
def recall_at_k(image_emb, text_emb, k=1):
    sims = np.matmul(text_emb, image_emb.T)
    topk = np.argsort(-sims, axis=1)[:, :k]
    correct = np.arange(len(text_emb))[:, None]
    hits = np.any(topk == correct, axis=1)
    return np.mean(hits)

print("Recall@1:", round(recall_at_k(val_img_emb, val_txt_emb, k=1), 4))
print("Recall@5:", round(recall_at_k(val_img_emb, val_txt_emb, k=5), 4))

## 12. Búsqueda texto → imagen

Dada una consulta en texto (p. ej. `"green triangle"`), la codificamos y recuperamos las imágenes de validación más parecidas por similitud, mostrando el *top-K* con su puntaje.

In [ ]:
def encode_query(text):
    token_ids = vectorizer(tf.constant([text]))
    emb = text_encoder(token_ids, training=False)
    emb = tf.math.l2_normalize(emb, axis=1).numpy()
    return emb

def search_images(query, top_k=5):
    q_emb = encode_query(query)
    sims = np.matmul(q_emb, val_img_emb.T)[0]
    top_idx = np.argsort(-sims)[:top_k]
    return top_idx, sims[top_idx]

query = "green triangle"
indices, scores = search_images(query, top_k=5)

print("Consulta:", query)
print("Índices:", indices)
print("Scores:", scores)

plt.figure(figsize=(12, 3))
for i, idx in enumerate(indices):
    plt.subplot(1, len(indices), i + 1)
    plt.imshow(val_images[idx])
    plt.axis("off")
    plt.title(f"Top {i+1}\n{scores[i]:.2f}")
plt.suptitle(f"Resultados para: {query}")
plt.tight_layout()
plt.show()